[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thebnbrkr/marv-audio/blob/main/notebooks/01_tutorial_whisper.ipynb)

# marv-audio tutorial: what Whisper listens to

Whisper turns speech into text in two halves. The **encoder** turns 30 seconds of
sound into 1,500 vectors (one per 20 ms). The **decoder** writes the transcript one
token at a time, and every decoder layer adds three things to a running sum (the
*residual stream*):

| write | reads |
|---|---|
| `self_attn` | the text written so far |
| `cross_attn` | the audio |
| `mlp` | whatever is already in the sum |

The final sum goes through a LayerNorm and becomes the scores for the next token.
Because it is a sum, each token's score can be split **exactly** into what each
write contributed, and MARV checks that the pieces add back up every time.

This notebook runs on a free Colab CPU in a few minutes. No GPU needed.

In [ ]:
# Installs marv-audio and MARV (from GitHub: the PyPI package called "marv" is unrelated).
!pip install -q "marv-audio[experiments] @ git+https://github.com/thebnbrkr/marv-audio"

## 1. Load whisper-tiny
Importing `marv_audio` teaches MARV where Whisper keeps its parts.

In [ ]:
import numpy as np, torch
import marv, marv_audio
from transformers import WhisperForConditionalGeneration, WhisperProcessor

MODEL, REVISION = "openai/whisper-tiny", "169d4a4341b33bc18d8881c4b69c2e104e1cc0af"  # pinned
processor = WhisperProcessor.from_pretrained(MODEL, revision=REVISION)
model = WhisperForConditionalGeneration.from_pretrained(MODEL, revision=REVISION, use_safetensors=True).eval()
tok = processor.tokenizer
print(type(marv.detect_adapter(model)).__name__, "| decoder layers:", len(model.model.decoder.layers))

## 2. Some speech
Three clips from LibriSpeech (audiobook recordings, with their reference text).

In [ ]:
from IPython.display import Audio, display

clips = marv_audio.librispeech_clips(3)
audio, reference = clips[0]
print(f"{len(audio) / 16000:.1f} s | reference: {reference}")
display(Audio(audio, rate=16000))

## 3. Transcribe it

Whisper's decoder starts from a fixed prefix: `<|startoftranscript|> <|en|>
<|transcribe|> <|notimestamps|>`. Then we pick the most likely next token until it
says `<|endoftext|>`. Every MARV call below takes the same `inputs` dict: the
audio features plus the decoder tokens.

In [ ]:
@torch.no_grad()
def transcribe(audio, max_new=100):
    ids = marv_audio.prompt_ids(processor)
    eot = tok.convert_tokens_to_ids("<|endoftext|>")
    for _ in range(max_new):
        logits = model(**marv_audio.whisper_inputs(processor, audio, ids)).logits[0, -1]
        ids.append(int(logits.argmax()))
        if ids[-1] == eot:
            break
    return ids

ids = transcribe(audio)
prefix_len = len(marv_audio.prompt_ids(processor))
print(tok.decode(ids[prefix_len:], skip_special_tokens=True))
inputs = marv_audio.whisper_inputs(processor, audio, ids)

## 4. The writes add up

`capture_writes` records every write at chosen positions. Their sum should equal
the real final residual. The check below is the guarantee behind every number
that follows; if it ever isn't ~0, don't trust the rest.

In [ ]:
w = marv.capture_writes(model, None, inputs, positions=list(range(len(ids))))
print("writes captured:", len(w.parts), "(4 layers x 3 parts)")
print(f"reconstruction error: {w.reconstruction_error():.1e}")

## 5. Why did it write this word?

`decompose_logit` splits one token's score (minus the average score) into the
embedding, every write of every layer, and the LayerNorm's bias. `check error`
compares the sum of the rows with the model's real output.

In [ ]:
j = prefix_len + 1                      # an early position in the transcript
target = ids[j + 1]
print("context:", repr(tok.decode(ids[prefix_len:j + 1])), "-> next:", repr(tok.decode([target])))
ctx = {"input_features": inputs["input_features"], "decoder_input_ids": inputs["decoder_input_ids"][:, :j + 1]}
d = marv.decompose_logit(model, None, ctx, target)
d.show(k=6);

## 6. Listening vs guessing, for every token at once

`listening_split` does the same split for every position in one pass. The
`cross_attn` column is the part that came **directly** from the audio.

In [ ]:
splits = marv_audio.listening_split(model, inputs, positions=list(range(prefix_len - 1, len(ids) - 1)))
print(f"max check error: {max(s.check_error for s in splits):.1e}\n")
print(f"{'token':>14} {'logit':>7} {'cross':>7} {'mlp':>7} {'self':>7}")
for s in splits:
    p = s.by_part
    print(f"{tok.decode([s.token])!r:>14} {s.actual:7.2f} {p['cross_attn']:7.2f} {p['mlp']:7.2f} {p['self_attn']:7.2f}")

The split above is **direct**: a write's own push on the score. Audio that a
cross-attention write hands to a later MLP is credited to the MLP. To measure the
audio's **total** effect, give the model one second of silence instead and see how
much less likely each token becomes (in nats: natural-log units of probability).

In [ ]:
@torch.no_grad()
def token_logprobs(audio):
    lp = torch.log_softmax(model(**marv_audio.whisper_inputs(processor, audio, ids)).logits[0].float(), -1)
    return np.array([float(lp[j, ids[j + 1]]) for j in range(prefix_len - 1, len(ids) - 1)])

effect = token_logprobs(audio) - token_logprobs(np.zeros(16000, dtype=np.float32))
order = np.argsort(effect)
words = [tok.decode([ids[j + 1]]) for j in range(prefix_len - 1, len(ids) - 1)]
print("needs the audio least:", [(words[i], round(float(effect[i]), 1)) for i in order[:5]])
print("needs the audio most: ", [(words[i], round(float(effect[i]), 1)) for i in order[-5:]])

In [ ]:
import matplotlib.pyplot as plt

cross = np.array([s.by_part["cross_attn"] for s in splits])
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(cross, effect)
for i in order[-3:]:
    ax.annotate(words[i].strip(), (cross[i], effect[i]))
ax.set_xlabel("direct cross-attention push (logit units)")
ax.set_ylabel("total audio effect (nats)")
ax.set_title("Each dot is one token")
plt.show()

## 7. Which language is it?

Whisper decides the language from the scores right after `<|startoftranscript|>`,
before any language token is in the input. `language_probs` reads exactly that,
normalised over all ~100 languages.

In [ ]:
probs = marv_audio.language_probs(model, processor, audio)
print({k: round(v, 3) for k, v in list(probs.items())[:5]})

## 8. Editing: switch off MLP neurons

A decoder MLP neuron is one column of `fc2`. `marv.suppress` zeroes chosen neurons
for the duration of a `with` block; nothing is changed permanently. Here we switch
off the 20 neurons that are most active at one position in the last layer, and see
how the probability of the word written there changes.

In [ ]:
layer = len(model.model.decoder.layers) - 1
acts = marv.feature_activations_at_layers(model, None, ctx, [layer])[layer]
top = np.argsort(-np.abs(acts))[:20]

@torch.no_grad()
def prob_of_target():
    return float(torch.softmax(model(**ctx).logits[0, -1].float(), -1)[target])

before = prob_of_target()
with marv.suppress(model, [(layer, int(f)) for f in top]):
    during = prob_of_target()
print(f"p({tok.decode([target])!r}): {before:.3f} -> {during:.3f} with 20 neurons off -> {prob_of_target():.3f} after")

## Where next

- `02_reproduce_experiments.ipynb` reruns E1 (listening vs guessing) and E2 (which
  writes carry the audio) and plots the results.
- `tests_colab.ipynb` runs the test suite plus exactness checks on the real model.
- `PREDICTIONS.md` in the repo has every experiment's predictions and outcomes.

Two rules this repo keeps: every decomposition ships its check, and **direct**
numbers (a write's own push) are never mixed up with **total** ones (what removing
something actually changes).